# Classify Behaviours

**Do not run this notebook in full** — it takes several hours. It is included for reproducibility: it shows exactly how the `behaviours{0,1,2}.npy` and `behaviours_normal{0,1,2}.npy` files were generated.

Uses atomic per-matrix checkpointing so it can be interrupted and resumed safely.

In [ ]:
# Must be set BEFORE importing numpy
import os
os.environ['OMP_NUM_THREADS']    = '1'
os.environ['MKL_NUM_THREADS']    = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pickle
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from numpy.fft import fft, fftfreq
from scipy.signal import find_peaks
from scipy.stats import entropy

sys.path.append('../../..')
sys.path.append('../../../results')

from utils import *

In [ ]:
data_root = '../../../data/fig2'
sigmas    = ['sigma02', 'sigma05', 'sigma1']

def load_matrices(path, key='W'):
    with open(path, 'rb') as f:
        data = pickle.load(f)
    if key == 'W':
        return [d['W'] for d in data if not np.all(d.get('W') == 0)]
    else:
        return [d[key] for d in data]

matrices_by_sigma        = {s: load_matrices(f'{data_root}/{s}/Ws.pkl')        for s in sigmas}
matrices_normal_by_sigma = {s: load_matrices(f'{data_root}/{s}/Ws_normal.pkl', key='W_normal') for s in sigmas}

In [ ]:
# --- Simulation parameters ---
N    = 100
T    = 300
dt   = 0.1
tauE = 1
tauI = 1
bias = np.zeros(N)
a    = np.ones(N)
c    = np.ones(N)
b    = np.zeros(N)

In [ ]:
# --- Per-matrix worker ---
def determine_behaviour(J):
    """Run all N initial conditions for one matrix and return a list of behaviour labels."""
    behaviours = []
    W, _ = W_from_J(J, a, b, c)
    ics  = init_cond(J)
    for i in range(N):
        x0 = 0.1 * ics[:, i]
        _, r = solve_rk4_springBox(x0, W, tauE, tauI, T, dt, a, b, c, bias)
        behaviour, _ = classify_behaviour(r, fft_tstart=50, fft_tend=300, dt=dt, n_pcs=5)
        behaviours.append(behaviour)
    return behaviours

In [ ]:
# --- Checkpointed parallel runner ---
def run_classification(matrices_dict, tag, out_dir, cache_dir='sweep_cache'):
    """
    Classify all matrices in matrices_dict with atomic per-matrix checkpointing.
    Saves behaviours{0,1,2}.npy (or behaviours_normal{0,1,2}.npy) to out_dir.

    Parameters
    ----------
    matrices_dict : dict sigma_key -> list of matrices
    tag           : str — 'nonnormal' or 'normal'
    out_dir       : str — where to save the .npy files
    """
    os.makedirs(cache_dir, exist_ok=True)

    # build flat job list tagged by sigma index and matrix index
    jobs = []
    for sigma_idx, (sigma_key, mats) in enumerate(matrices_dict.items()):
        for mat_idx, J in enumerate(mats):
            jobs.append((sigma_idx, sigma_key, mat_idx, J))
    print(f'Total matrices ({tag}): {len(jobs)}')

    def cache_path(sigma_idx, mat_idx):
        return f'{cache_dir}/{tag}_s{sigma_idx}_m{mat_idx}.pkl'

    def process_one(sigma_idx, sigma_key, mat_idx, J):
        path = cache_path(sigma_idx, mat_idx)
        if os.path.exists(path):
            return  # already done
        result = determine_behaviour(J)
        with open(path, 'wb') as f:
            pickle.dump(result, f)

    Parallel(n_jobs=-1, backend='loky', verbose=0)(
        delayed(process_one)(si, sk, mi, J)
        for si, sk, mi, J in tqdm(jobs, desc=tag)
    )

    # reassemble and save per sigma
    suffix = '' if tag == 'nonnormal' else '_normal'
    for sigma_idx, (sigma_key, mats) in enumerate(matrices_dict.items()):
        results = []
        for mat_idx in range(len(mats)):
            with open(cache_path(sigma_idx, mat_idx), 'rb') as f:
                results.append(pickle.load(f))
        arr = np.array(results, dtype=object)
        out_path = f'{out_dir}/behaviours{suffix}{sigma_idx}.npy'
        np.save(out_path, arr)
        print(f'Saved {out_path} ({len(results)} matrices)')

In [ ]:
# --- Run classification (non-normal) ---
# WARNING: takes several hours
run_classification(
    matrices_by_sigma,
    tag='nonnormal',
    out_dir=f'{data_root}/behaviours'
)

In [ ]:
# --- Run classification (normal) ---
# WARNING: takes several hours
run_classification(
    matrices_normal_by_sigma,
    tag='normal',
    out_dir=f'{data_root}/behaviours'
)

In [ ]:
# --- Clear cache after successful completion ---
import shutil
shutil.rmtree('sweep_cache', ignore_errors=True)
print('Cache cleared.')